In [1]:
# Read the orders JSON file
# multiline=true because the file is a JSON array, not one record per line
orders_df = spark.read \
    .option("multiline", "true") \
    .json("Files/raw/orders/orders_20240101_20240131.json")

print("Orders row count:", orders_df.count())
print("Orders columns:", orders_df.columns)
orders_df.show(3, truncate=50)

StatementMeta(, 861dcc54-9d59-401e-aefb-8650bd6f7f82, 3, Finished, Available, Finished, False)

Orders row count: 1400
Orders columns: ['city', 'customer_id', 'discount_code', 'items', 'order_date', 'order_id', 'order_status', 'order_total', 'payment_method', 'state']
+---------+-----------+-------------+--------------------------------------------------+-------------------+----------+------------+-----------+--------------+-----+
|     city|customer_id|discount_code|                                             items|         order_date|  order_id|order_status|order_total|payment_method|state|
+---------+-----------+-------------+--------------------------------------------------+-------------------+----------+------------+-----------+--------------+-----+
|    Delhi|  CUST10172|       SAVE24|[{Toys, 0.2, ITEM81482, 9855.28, PROD1022, 1, 1...|2024-01-01T20:07:01|ORD0000001|   delivered|   42610.12|    Debit Card|   KA|
|   Jaipur|  CUST10327|       SAVE11|[{Books, 0.19, ITEM57400, 6530.43, PROD1147, 2,...|2024-01-01T17:07:59|ORD0000002|     shipped|  104337.31|        Wallet|   D

In [2]:
# Read the customers CSV file
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("Files/raw/customers/customers_master.csv")

print("Customers row count:", customers_df.count())
print("Customers columns:", customers_df.columns)
customers_df.show(3, truncate=50)

StatementMeta(, 861dcc54-9d59-401e-aefb-8650bd6f7f82, 4, Finished, Available, Finished, False)

Customers row count: 500
Customers columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'state', 'pincode', 'date_joined', 'loyalty_tier', 'total_orders', 'is_active']
+-----------+----------+---------+--------------------------+-------------+-------+-----+-------+-----------+------------+------------+---------+
|customer_id|first_name|last_name|                     email|        phone|   city|state|pincode|date_joined|loyalty_tier|total_orders|is_active|
+-----------+----------+---------+--------------------------+-------------+-------+-----+-------+-----------+------------+------------+---------+
|  CUST10000|     Arjun|     Nair|arjun.nair0@rediffmail.com|+918647547307| Mumbai|   DL| 103801| 2022-11-19|    Platinum|          24|      Yes|
|  CUST10001|     Meera|    Singh|    meera.singh1@gmail.com|+918050062761|Kolkata|   MH| 632250| 2021-02-22|        Gold|         141|     TRUE|
|  CUST10002|    Vikram|    Gupta| vikram.gupta2@hotmail.com|+917576946240|  

In [3]:
# Products file has a wrapper: {"data": [...], "total": 200}
# So we read it, then extract the "data" array
products_raw = spark.read \
    .option("multiline", "true") \
    .json("Files/raw/products/products_catalog.json")

print("Top level columns:", products_raw.columns)
products_raw.show(2, truncate=80)

StatementMeta(, 861dcc54-9d59-401e-aefb-8650bd6f7f82, 5, Finished, Available, Finished, False)

Top level columns: ['data', 'fetched_at', 'total']
+--------------------------------------------------------------------------------+--------------------------+-----+
|                                                                            data|                fetched_at|total|
+--------------------------------------------------------------------------------+--------------------------+-----+
|[{19684.23, Apple, Books, 2023-01-31T00:00:00, true, PROD1000, Noise Books Mo...|2026-07-01T18:21:04.621843|  200|
+--------------------------------------------------------------------------------+--------------------------+-----+



In [4]:
# Read inventory CSV
inventory_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("Files/raw/inventory/inventory_snapshot.csv")

print("Inventory row count:", inventory_df.count())
print("Inventory columns:", inventory_df.columns)
inventory_df.show(3, truncate=50)

StatementMeta(, 861dcc54-9d59-401e-aefb-8650bd6f7f82, 6, Finished, Available, Finished, False)

Inventory row count: 610
Inventory columns: ['product_id', 'warehouse_code', 'quantity_available', 'quantity_reserved', 'reorder_threshold', 'last_updated']
+----------+--------------+------------------+-----------------+-----------------+-------------------+
|product_id|warehouse_code|quantity_available|quantity_reserved|reorder_threshold|       last_updated|
+----------+--------------+------------------+-----------------+-----------------+-------------------+
|  PROD1000|     WH_MUM_01|               155|               35|               50|2026-07-01 18:21:04|
|  PROD1000|     WH_DEL_01|               300|               45|               20|2026-07-01 18:21:04|
|  PROD1000|     WH_HYD_01|                91|                3|               10|2026-07-01 18:21:04|
+----------+--------------+------------------+-----------------+-----------------+-------------------+
only showing top 3 rows



In [5]:
# Print a clean summary of all four files
print("=" * 40)
print("UPLOAD VERIFICATION SUMMARY")
print("=" * 40)

files = {
    "Orders": ("Files/raw/orders/orders_20240101_20240131.json", "json"),
    "Customers": ("Files/raw/customers/customers_master.csv", "csv"),
    "Inventory": ("Files/raw/inventory/inventory_snapshot.csv", "csv"),
}

for name, (path, fmt) in files.items():
    if fmt == "json":
        df = spark.read.option("multiline", "true").json(path)
    else:
        df = spark.read.option("header", "true").csv(path)
    print(f"{name}: {df.count()} rows, {len(df.columns)} columns")

# Products separately due to nested structure
products_raw = spark.read.option("multiline","true").json(
    "Files/raw/products/products_catalog.json"
)
total = products_raw.select("total").collect()[0][0]
print(f"Products: {total} products in catalog")

print("=" * 40)
print("All files verified. Ready for Bronze ingestion.")

StatementMeta(, 861dcc54-9d59-401e-aefb-8650bd6f7f82, 7, Finished, Available, Finished, False)

UPLOAD VERIFICATION SUMMARY
Orders: 1400 rows, 10 columns
Customers: 500 rows, 12 columns
Inventory: 610 rows, 6 columns
Products: 200 products in catalog
All files verified. Ready for Bronze ingestion.
